# ЛР4 — отложенная проверка ReAct и Plan-then-execute

Новая сессия Kaggle. Выберите GPU T4 и включите Internet. Запускайте ячейки по порядку. Блокнот использует зафиксированные 12 отложенных задач учебного календаря и документов (24 эпизода), конфигурацию v2 и локальный сервер vLLM. При ошибках результаты сохраняются отдельно.

In [ ]:
import os, sys, json, shutil, subprocess, time
from pathlib import Path

ART = Path("/kaggle/working/lab4_session")
ART.mkdir(parents=True, exist_ok=True)
REPO = Path("/tmp/lab4-project-final")
COMMIT = "6a5929b976d15e2dcf4f73bc2cfe2a994141ec7a"

ENV = dict(os.environ)
ENV.pop("PYTHONPATH", None)
ENV.update(CUDA_VISIBLE_DEVICES="0", HF_HOME="/tmp/lab4-hf",
           PYTHONNOUSERSITE="1", PYTHONUNBUFFERED="1",
           OMP_NUM_THREADS="1", TOKENIZERS_PARALLELISM="false")

def cmd(args, **kwargs):
    return subprocess.run([str(arg) for arg in args], env=ENV, check=True, **kwargs)

gpu = cmd(["nvidia-smi", "--id=0", "--query-gpu=name", "--format=csv,noheader"],
          capture_output=True, text=True).stdout.strip()
assert "T4" in gpu, gpu
print("GPU:", gpu)

if not REPO.exists():
    cmd(["git", "clone", "--depth", "1", "--filter=blob:none", "--sparse",
         "--single-branch", "--branch", "main",
         "https://github.com/manaer6-alt/intsys-labs.git", REPO])
    cmd(["git", "-C", REPO, "sparse-checkout", "set", "lab4", "data/agentdojo"])
head = cmd(["git", "-C", REPO, "rev-parse", "HEAD"], capture_output=True, text=True).stdout.strip()
if head != COMMIT:
    cmd(["git", "-C", REPO, "fetch", "--depth", "1", "origin", COMMIT])
    cmd(["git", "-C", REPO, "checkout", "--detach", COMMIT])
assert cmd(["git", "-C", REPO, "rev-parse", "HEAD"],
           capture_output=True, text=True).stdout.strip() == COMMIT
assert (REPO / "lab4/run_final.py").exists()
print("OK: code commit", COMMIT)
(ART / "commit.txt").write_text(COMMIT + "\n", encoding="utf-8")


In [ ]:
# Два отдельных окружения: сервер и учебный бенчмарк.
if not shutil.which("uv"):
    cmd([sys.executable, "-m", "pip", "install", "uv"])
uv = shutil.which("uv")
assert uv

PYTHON = "/tmp/lab4-env/bin/python"
AGENT_PYTHON = "/tmp/lab4-agent-env/bin/python"
if not Path(PYTHON).exists():
    cmd([uv, "venv", "--python", "3.13", "--seed", "/tmp/lab4-env"])
cmd([uv, "pip", "install", "--python", PYTHON, "--torch-backend=auto",
     "vllm==0.33.0", "torch==2.14.0+cu132", "transformers==5.20.2",
     "huggingface-hub==2.4.0", "tokenizers==0.23.3", "numpy==2.4.6",
     "openai==3.45.0", "PyYAML==6.0.3", "wrapt==2.5.2"])
cmd([uv, "pip", "check", "--python", PYTHON])
cmd([PYTHON, "-c", "import torch; assert torch.cuda.is_available() and torch.cuda.device_count()==1; print(torch.cuda.get_device_name(0))"])

if not Path(AGENT_PYTHON).exists():
    cmd([uv, "venv", "--python", "3.13", "--seed", "/tmp/lab4-agent-env"])
cmd([uv, "pip", "install", "--python", AGENT_PYTHON,
     "agentdojo==0.1.35", "wrapt==2.5.2"])
cmd([uv, "pip", "check", "--python", AGENT_PYTHON])
cmd([AGENT_PYTHON, "-c", "from agentdojo.task_suite.load_suites import get_suite; s=get_suite('v1.2.2','workspace'); assert len(s.user_tasks)==40; print('OK: 40 tasks')"])

for python, filename in ((PYTHON, "requirements_gpu.txt"), (AGENT_PYTHON, "requirements_agent.txt")):
    freeze = cmd([uv, "pip", "freeze", "--python", python], capture_output=True, text=True)
    (ART / filename).write_text(freeze.stdout, encoding="utf-8")
preflight = "from run_final import prepare; data, numbers=prepare(); assert len(numbers)==12; print('Frozen selection:',numbers)"
cmd([AGENT_PYTHON, "-B", "-c", preflight], cwd=REPO / "lab4")


In [ ]:
# Запускаем Qwen на одной T4 только на локальном интерфейсе.
import urllib.request

MODEL = "Qwen/Qwen3-4B-Instruct-2507"
REV = "cdbee75f17c01a7cc42f958dc650907174af0554"
BASE = "http://127.0.0.1:8000"
command = ["/tmp/lab4-env/bin/vllm", "serve", MODEL,
           "--revision", REV, "--tokenizer-revision", REV,
           "--dtype", "half", "--max-model-len", "16384",
           "--max-num-seqs", "1", "--max-num-batched-tokens", "2048",
           "--gpu-memory-utilization", "0.9", "--tensor-parallel-size", "1",
           "--attention-backend", "TRITON_ATTN", "--generation-config", "vllm",
           "--enable-auto-tool-choice", "--tool-call-parser", "hermes",
           "--no-enable-prefix-caching", "--seed", "0",
           "--host", "127.0.0.1", "--port", "8000"]
if "server" not in globals() or server.poll() is not None:
    with (ART / "server.log").open("w", encoding="utf-8") as log:
        server = subprocess.Popen(command, env=ENV, stdout=log, stderr=subprocess.STDOUT)
(ART / "server_config.json").write_text(json.dumps({
    "model": MODEL, "revision": REV, "gpu": gpu,
    "base_url": BASE + "/v1", "command": command}, indent=2), encoding="utf-8")
for i in range(180):
    if server.poll() is not None:
        raise RuntimeError((ART / "server.log").read_text(encoding="utf-8", errors="replace")[-6000:])
    try:
        with urllib.request.urlopen(BASE + "/v1/models", timeout=3) as response:
            assert MODEL in [item["id"] for item in json.load(response)["data"]]
            print("Server ready")
            break
    except OSError:
        time.sleep(5)
else:
    raise TimeoutError("Server startup timeout. See server.log")


In [ ]:
# Одна отдельная проверка формата tool_calls, не входит в итоговые эпизоды.
tools = [{"type": "function", "function": {"name": "get_current_day",
    "description": "Return the current date in the simulated calendar.",
    "parameters": {"type": "object", "properties": {}, "additionalProperties": False}}}]
body = {"model": MODEL, "messages": [
    {"role": "system", "content": "Use the provided calendar tool."},
    {"role": "user", "content": "What is today's date in the simulated calendar?"}],
    "tools": tools, "tool_choice": "auto", "temperature": 0,
    "top_p": 1, "seed": 0, "max_tokens": 1024}
request = urllib.request.Request(BASE + "/v1/chat/completions",
    data=json.dumps(body).encode(), headers={"Content-Type": "application/json"})
with urllib.request.urlopen(request, timeout=180) as response:
    smoke = json.load(response)
(ART / "tool_call_check.json").write_text(json.dumps(smoke, indent=2), encoding="utf-8")
calls = smoke["choices"][0]["message"].get("tool_calls") or []
assert len(calls) == 1 and calls[0]["function"]["name"] == "get_current_day"
assert json.loads(calls[0]["function"]["arguments"]) == {}
print("OK: structured tool_calls")


In [ ]:
# Основная ограниченная отложенная серия: 12 задач x 2 режима.
OUT = Path("/kaggle/working/lab4_final")
assert not OUT.exists(), "Каталог уже существует. Не запускайте итоговую серию повторно."
cmd([AGENT_PYTHON, "-B", REPO / "lab4/run_final.py", "--out", OUT])
assert (OUT / "final_summary.json").is_file()
summary = json.loads((OUT / "final_summary.json").read_text(encoding="utf-8"))
assert summary["episodes"] == 24
print("OK: 24 episodes", summary["pairs"])


In [ ]:
# Сохраняем результаты и метаданные сессии. Рабочие файлы также остаются в Output.
OUT = Path("/kaggle/working/lab4_final")
assert OUT.is_dir()
for name in ("commit.txt", "server_config.json", "requirements_gpu.txt",
             "requirements_agent.txt", "tool_call_check.json"):
    shutil.copy2(ART / name, OUT / name)
complete = (OUT / "final_summary.json").is_file()
basename = "/kaggle/working/lab4_final_results" if complete else "/kaggle/working/lab4_final_partial"
archive = shutil.make_archive(basename, "zip", OUT)
print("Download:", archive)
if "server" in globals() and server.poll() is None:
    server.terminate()
    try:
        server.wait(timeout=30)
    except subprocess.TimeoutExpired:
        server.kill()
        server.wait()
